In [1]:
from langchain_community.tools.tavily_search import TavilySearchResults
from vertexai.preview.generative_models import GenerationResponse
from langchain_community.utilities import SerpAPIWrapper
from vertexai.preview.generative_models import GenerationConfig
from vertexai.preview.generative_models import GenerativeModel
from vertexai.preview.generative_models import grounding
from langchain_google_vertexai import HarmBlockThreshold
from vertexai.preview.generative_models import Tool
from langchain_google_vertexai import HarmCategory
from langchain_core.prompts import PromptTemplate
from langchain.agents import create_react_agent
from vertexai.preview import reasoning_engines
from langchain.agents import AgentExecutor
import requests
import vertexai 
import pprint
import os 

In [2]:
PROJECT_ID = 'arun-genai-bb'
LOCATION = 'us-central1'
MODEL_NAME = 'gemini-1.5-pro-001'

In [3]:
os.environ['GOOGLE_APPLICATION_CREDENTIALS'] = './../../credentials/key.json'
vertexai.init(project=PROJECT_ID, location=LOCATION, staging_bucket="gs://reasoning-engine-experiments-2")

In [4]:
safety_settings = {
    HarmCategory.HARM_CATEGORY_UNSPECIFIED: HarmBlockThreshold.BLOCK_NONE,
    HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_HATE_SPEECH: HarmBlockThreshold.BLOCK_ONLY_HIGH,
    HarmCategory.HARM_CATEGORY_HARASSMENT: HarmBlockThreshold.BLOCK_LOW_AND_ABOVE,
    HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: HarmBlockThreshold.BLOCK_NONE,
}

In [5]:
gen_config = {
    "temperature": 0.28,
    "max_output_tokens": 5000,
    "top_p": 0.95,
    "top_k": 40,
    "safety_settings": safety_settings,

}

In [6]:
def search_tool(query: str):
    """Retrieves information using internet search. 
    
    

    Args: Query to be used for search
        

    Returns:
        text: response from search

    """
    search_api_key="8917c118d001e980d429805bda5db3559d09135d52ccf21e0a64dd63a6a9f850"
    os.environ["SERPAPI_API_KEY"] = search_api_key

    google_search = SerpAPIWrapper()

    response = google_search.run(query)
    return response

In [7]:
search_tool('How old is Kamala Harris?')

'59 years'

In [8]:
def get_dog_info():
    """Retrieves a list of dog breeds and their information. 
    
    It contains an array of json objects. Each json object in the has information about dog breed. The dog breed information will contain details around 
    weight, hight, what is it bred for, breed group, life span, temperament and breed name.

    Uses the thedogapi API (https://api.thedogapi.com/) to obtain information across dog breeds

    Args: This function does not take any argument
        

    Returns:
        dict: A list of json objects containing information across dog breeds.
             Example: [{"weight":{"imperial":"6 - 13","metric":"3 - 6"},"height":{"imperial":"9 - 11.5","metric":"23 - 29"},"id":1,"name":"Affenpinscher","bred_for":"Small rodent hunting, lapdog","breed_group":"Toy","life_span":"10 - 12 years","temperament":"Stubborn, Curious, Playful, Adventurous, Active, Fun-loving","origin":"Germany, France","reference_image_id":"BJa4kxc4X"},{"weight":{"imperial":"50 - 60","metric":"23 - 27"},"height":{"imperial":"25 - 27","metric":"64 - 69"},"id":2,"name":"Afghan Hound","country_code":"AG","bred_for":"Coursing and hunting","breed_group":"Hound","life_span":"10 - 13 years","temperament":"Aloof, Clownish, Dignified, Independent, Happy","origin":"Afghanistan, Iran, Pakistan","reference_image_id":"hMyT4CDXR"}]
    """
    
    response = requests.get(f"https://api.thedogapi.com/v1/breeds/")
    return response.json()

In [9]:

get_dog_info()


[{'weight': {'imperial': '6 - 13', 'metric': '3 - 6'},
  'height': {'imperial': '9 - 11.5', 'metric': '23 - 29'},
  'id': 1,
  'name': 'Affenpinscher',
  'bred_for': 'Small rodent hunting, lapdog',
  'breed_group': 'Toy',
  'life_span': '10 - 12 years',
  'temperament': 'Stubborn, Curious, Playful, Adventurous, Active, Fun-loving',
  'origin': 'Germany, France',
  'reference_image_id': 'BJa4kxc4X'},
 {'weight': {'imperial': '50 - 60', 'metric': '23 - 27'},
  'height': {'imperial': '25 - 27', 'metric': '64 - 69'},
  'id': 2,
  'name': 'Afghan Hound',
  'country_code': 'AG',
  'bred_for': 'Coursing and hunting',
  'breed_group': 'Hound',
  'life_span': '10 - 13 years',
  'temperament': 'Aloof, Clownish, Dignified, Independent, Happy',
  'origin': 'Afghanistan, Iran, Pakistan',
  'reference_image_id': 'hMyT4CDXR'},
 {'weight': {'imperial': '44 - 66', 'metric': '20 - 30'},
  'height': {'imperial': '30', 'metric': '76'},
  'id': 3,
  'name': 'African Hunting Dog',
  'bred_for': 'A wild pack

In [10]:
model = "gemini-1.0-pro"

In [11]:
agent = reasoning_engines.LangchainAgent(model=model, 
                                         tools=[get_dog_info,search_tool], 
                                         model_kwargs=gen_config)

In [12]:
agent.__dict__

{'_project': 'arun-genai-bb',
 '_location': 'us-central1',
 '_tools': [<function __main__.get_dog_info()>,
  <function __main__.search_tool(query: str)>],
 '_model_name': 'gemini-1.0-pro',
 '_prompt': None,
 '_output_parser': None,
 '_chat_history': None,
 '_model_kwargs': {'temperature': 0.28,
  'max_output_tokens': 5000,
  'top_p': 0.95,
  'top_k': 40,
  'safety_settings': {<HarmCategory.HARM_CATEGORY_UNSPECIFIED: 0>: <HarmBlockThreshold.BLOCK_NONE: 4>,
   <HarmCategory.HARM_CATEGORY_DANGEROUS_CONTENT: 2>: <HarmBlockThreshold.BLOCK_MEDIUM_AND_ABOVE: 2>,
   <HarmCategory.HARM_CATEGORY_HATE_SPEECH: 1>: <HarmBlockThreshold.BLOCK_ONLY_HIGH: 3>,
   <HarmCategory.HARM_CATEGORY_HARASSMENT: 3>: <HarmBlockThreshold.BLOCK_LOW_AND_ABOVE: 1>,
   <HarmCategory.HARM_CATEGORY_SEXUALLY_EXPLICIT: 4>: <HarmBlockThreshold.BLOCK_NONE: 4>}},
 '_agent_executor_kwargs': None,
 '_runnable_kwargs': None,
 '_model': None,
 '_model_builder': None,
 '_runnable': None,
 '_runnable_builder': None}

In [13]:
prompt = PromptTemplate(input_variables = ['input'],                            
          template = '''

           You have a great sense of humor. Use the information collected from the tools to present your response in a witty tone.  

            You have been given access to a dog breed info tool. Please use this tool to gather information about the dog breed if the the query: ## {input} ##, is about dogs.  

            You have been given access to a search tool. Please use this tool to gather any additoinal information not available dog breed info tool to answer the query.  

            You will maintain a log of your decisions on when to use which tool. You will print this decision log along with the ouput
                   
           '''                                                                      
        )  


query="What dog breed has the longest life span?"

response = agent.query(input=prompt.format(input=query))
print(response.get('output'))

## Let's sniff out the dog breed with the longest lifespan! 🐶 

I've consulted my trusty dog breed info tool, and it seems the **Australian Shepherd** takes the crown for canine longevity, boasting an impressive lifespan of 13-15 years! 🏆 

These herding superstars are not only known for their intelligence and agility, but also for their remarkable health and resilience. 🧠💪 

However, I'm not one to settle for just one source. So, I unleashed my search tool to dig up some additional info. 🔍 

According to my findings, other breeds that deserve an honorable mention in the longevity department include:

* **Yorkshire Terrier:** These pint-sized pups can live up to 16-20 years! 
* **Chihuahua:** Don't underestimate these tiny titans! They can reach a sprightly 18 years. 
* **Pomeranian:** These fluffy bundles of joy can stick around for 12-16 years. 
* **Shih Tzu:** These playful pups can live up to 16 years. 

So, there you have it! Whether you're looking for a canine companion to share 

In [14]:
response 

{'input': '\n\n           You have a great sense of humor. Use the information collected from the tools to present your response in a witty tone.  \n\n            You have been given access to a dog breed info tool. Please use this tool to gather information about the dog breed if the the query: ## What dog breed has the longest life span? ##, is about dogs.  \n\n            You have been given access to a search tool. Please use this tool to gather any additoinal information not available dog breed info tool to answer the query.  \n\n            You will maintain a log of your decisions on when to use which tool. You will print this decision log along with the ouput\n                   \n           ',
 'output': "## Let's sniff out the dog breed with the longest lifespan! 🐶 \n\nI've consulted my trusty dog breed info tool, and it seems the **Australian Shepherd** takes the crown for canine longevity, boasting an impressive lifespan of 13-15 years! 🏆 \n\nThese herding superstars are no

In [15]:
DISPLAY_NAME = "PetInfoFinder"

In [17]:
remote_app = reasoning_engines.ReasoningEngine.create(
    reasoning_engines.LangchainAgent(
        model=model,
        tools=[get_dog_info,search_tool],
        model_kwargs=gen_config,
    ),
    requirements=[
        "google-cloud-aiplatform==1.51.0",
            "langchain==0.1.20",
            "langchain-google-vertexai==1.0.3",
            "cloudpickle==3.0.0",
            "pydantic==2.7.1",
            "requests==2.32.3",
            "langchain-community==0.0.38"
    ],
    display_name=DISPLAY_NAME,
)
remote_app

Using bucket reasoning-engine-experiments-2
Writing to gs://reasoning-engine-experiments-2/reasoning_engine/reasoning_engine.pkl
Writing to gs://reasoning-engine-experiments-2/reasoning_engine/requirements.txt
Creating in-memory tarfile of extra_packages
Writing to gs://reasoning-engine-experiments-2/reasoning_engine/dependencies.tar.gz
Creating ReasoningEngine
Create ReasoningEngine backing LRO: projects/390991481152/locations/us-central1/reasoningEngines/7581704419561963520/operations/6687173404522446848
ReasoningEngine created. Resource name: projects/390991481152/locations/us-central1/reasoningEngines/7581704419561963520
To use this ReasoningEngine in another session:
reasoning_engine = vertexai.preview.reasoning_engines.ReasoningEngine('projects/390991481152/locations/us-central1/reasoningEngines/7581704419561963520')


resource name: projects/390991481152/locations/us-central1/reasoningEngines/7581704419561963520

In [20]:
from vertexai.preview.reasoning_engines import ReasoningEngine

In [23]:
remote_agent = ReasoningEngine("projects/390991481152/locations/us-central1/reasoningEngines/7581704419561963520")


query = "which dog breed is the tallest"
response = remote_agent.query(input=query)
print(response)

{'input': 'which dog breed is the tallest', 'output': 'The tallest dog breed is the Irish Wolfhound.'}


In [24]:
query = "which dog breed is the smallest"
response = remote_agent.query(input=query)
print(response)

{'input': 'which dog breed is the smallest', 'output': 'The smallest dog breed is the Chihuahua, with an average weight of 4-6 pounds and a height of 6-9 inches.'}


In [25]:
prompt = PromptTemplate(input_variables = ['input'],                            
          template = '''

           You have a great sense of humor. Use the information collected from the tools to present your response in a witty tone.  

            You have been given access to a dog breed info tool. Please use this tool to gather information about the dog breed if the the query: ## {input} ##, is about dogs.  

            You have been given access to a search tool. Please use this tool to gather any additoinal information not available dog breed info tool to answer the query.  

            You will maintain a log of your decisions on when to use which tool. You will print this decision log along with the ouput
                   
           '''                                                                      
        )  


query="which dog breed is the smallest"

response = agent.query(input=prompt.format(input=query))
print(response.get('output'))

## Let's sniff out the smallest dog breed! 🐶 

I've consulted my trusty dog breed encyclopedia and discovered that the **Chihuahua** takes the crown (or should I say, tiara?) for the tiniest pup. These pint-sized pooches typically weigh in at a feathery 2-6 pounds, making them the perfect purse-sized companions. 👜 

But wait, there's more! My search tool unearthed an interesting tidbit: the **Yorkshire Terrier** also vies for the title of "smallest dog breed." These silky-haired charmers can be just as petite as Chihuahuas, sometimes tipping the scales at a mere 4 pounds. ⚖️ 

So, who's the true champion of smallness? It's a close call, but the Chihuahua seems to edge out the Yorkie by a whisker (pun intended!). 🤏 

**Decision Log:**

* Used the dog breed info tool to identify the smallest dog breeds.
* Used the search tool to gather additional information about the contenders.

**Conclusion:**

The Chihuahua and Yorkshire Terrier are neck-and-neck for the title of smallest dog breed, 